# Selected lncRNA Validation

Per-gene analysis of 6 hand-picked lncRNAs (MALAT1, NEAT1, XIST, H19, XACT, KCNQ1OT1)
projected from human (hg38) to mouse (mm39).

- Do predicted query islands overlap known mouse GENCODE exons?
- What are the MMD scores per island?
- Per-gene breakdown with plots.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

pyrion_path = Path("/Users/Bogdan.Kirilenko/PycharmProjects/pyrion")
if pyrion_path not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(pyrion_path))

from pyrion.io.bed import read_bed12_file
from pyrion.ops.interval_ops import intersect_intervals, merge_intervals
from pyrion.core.intervals import GenomicInterval

In [ ]:
SELECTED_GENES = {
    "ENSG00000251562": "MALAT1",
    "ENSG00000245532": "NEAT1",
    "ENSG00000229807": "XIST",
    "ENSG00000130600": "H19",
    "ENSG00000241743": "XACT",
    "ENSG00000269821": "KCNQ1OT1",
}
SELECTED_IDS = {f"U_{gid}" for gid in SELECTED_GENES}
ID_TO_NAME = {f"U_{gid}": name for gid, name in SELECTED_GENES.items()}

PIPELINE_DIR = Path("../quick_test")
ANNOTATION_DIR = Path("../input_data/mm39_annotation_validation")

results = pd.read_csv(PIPELINE_DIR / "island_alignment_results.tsv", sep="\t")
query_bed = read_bed12_file(str(PIPELINE_DIR / "intermediate_bed_files" / "aligned_islands_query.bed"))
ref_bed = read_bed12_file(str(PIPELINE_DIR / "intermediate_bed_files" / "aligned_islands_reference.bed"))
annotation = read_bed12_file(str(ANNOTATION_DIR / "mm39_quick_test_transcripts.bed"))

sel = results[results["gene_id"].isin(SELECTED_IDS)].copy()
sel["gene_name"] = sel["gene_id"].map(ID_TO_NAME)

print(f"Island results: {len(results)} total, {len(sel)} for selected genes")
print(f"Query BED: {len(query_bed)} entries")
print(f"Annotation: {len(annotation):,} transcripts")
print(f"\nSelected genes: {', '.join(ID_TO_NAME.values())}")

## Overview

In [ ]:
overview_rows = []
for gid, name in sorted(ID_TO_NAME.items(), key=lambda x: x[1]):
    g = sel[sel["gene_id"] == gid]
    q_transcript = query_bed.get_by_id(f"{gid}_aligned")
    q_span = f"{q_transcript.chrom}:{q_transcript.start}-{q_transcript.end}" if q_transcript else "N/A"
    q_bp = int(np.sum(q_transcript.blocks[:, 1] - q_transcript.blocks[:, 0])) if q_transcript else 0
    overview_rows.append({
        "gene": name,
        "gene_id": gid,
        "query_location": q_span,
        "n_islands": len(g),
        "n_anchor": (g["type"] == "anchor").sum(),
        "n_fill": (g["type"] == "fill").sum(),
        "query_span_bp": q_transcript.end - q_transcript.start if q_transcript else 0,
        "query_exonic_bp": q_bp,
        "median_mmd": g["diag_mmd"].median(),
        "mean_mmd": g["diag_mmd"].mean(),
    })

overview = pd.DataFrame(overview_rows)
print(overview[["gene", "query_location", "n_islands", "n_anchor", "n_fill",
                 "query_exonic_bp", "median_mmd"]].to_string(index=False))

## Overlap with mouse GENCODE annotation

In [ ]:
overlap_rows = []
for gid, name in sorted(ID_TO_NAME.items(), key=lambda x: x[1]):
    q_t = query_bed.get_by_id(f"{gid}_aligned")
    if q_t is None:
        continue

    q_interval = GenomicInterval(q_t.chrom, q_t.start, q_t.end)
    hits = annotation.get_transcripts_in_interval(q_interval)

    best_bp = 0
    best_pct = 0.0
    best_tid = None
    exon_hit_ids = []
    q_exonic_bp = int(np.sum(q_t.blocks[:, 1] - q_t.blocks[:, 0]))

    for t in hits:
        isect = intersect_intervals(q_t.blocks, t.blocks)
        if len(isect) > 0:
            bp = int(np.sum(isect[:, 1] - isect[:, 0]))
            exon_hit_ids.append((t.id, bp))
            if bp > best_bp:
                best_bp = bp
                best_pct = bp / q_exonic_bp * 100
                best_tid = t.id

    overlap_rows.append({
        "gene": name,
        "query_chrom": q_t.chrom,
        "query_exonic_bp": q_exonic_bp,
        "n_transcripts_in_span": len(hits),
        "n_exon_overlapping": len(exon_hit_ids),
        "best_overlap_bp": best_bp,
        "best_overlap_pct": best_pct,
        "best_transcript": best_tid,
        "all_exon_hits": exon_hit_ids,
    })

overlap_df = pd.DataFrame(overlap_rows)
print(overlap_df[["gene", "query_exonic_bp", "n_transcripts_in_span",
                    "n_exon_overlapping", "best_overlap_bp", "best_overlap_pct",
                    "best_transcript"]].to_string(index=False))

print("\n--- Detailed exon-level hits ---")
for _, row in overlap_df.iterrows():
    if row["all_exon_hits"]:
        print(f"\n{row['gene']}:")
        for tid, bp in sorted(row["all_exon_hits"], key=lambda x: -x[1])[:5]:
            print(f"  {tid}: {bp} bp overlap")
    else:
        print(f"\n{row['gene']}: no exon overlap")

## MMD score distributions

In [ ]:
gene_names_sorted = sorted(ID_TO_NAME.values())
n_genes = len(gene_names_sorted)
name_to_id = {v: k for k, v in ID_TO_NAME.items()}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, name in enumerate(gene_names_sorted):
    ax = axes[i]
    gid = name_to_id[name]
    g = sel[sel["gene_id"] == gid]

    anchors = g[g["type"] == "anchor"]["diag_mmd"]
    fills = g[g["type"] == "fill"]["diag_mmd"]

    if len(anchors) > 0:
        ax.hist(anchors, bins=15, alpha=0.7, color="#42A5F5", label=f"anchor (n={len(anchors)})",
                edgecolor="white")
    if len(fills) > 0:
        ax.hist(fills, bins=15, alpha=0.7, color="#FFA726", label=f"fill (n={len(fills)})",
                edgecolor="white")

    ax.axvline(g["diag_mmd"].median(), color="red", linestyle="--", linewidth=1,
               label=f"median={g['diag_mmd'].median():.3f}")
    ax.set_title(name, fontweight="bold")
    ax.set_xlabel("diag_mmd")
    ax.set_ylabel("Count")
    ax.legend(fontsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## Per-gene detail

For each gene: island layout on query genome, with GENCODE exon overlap shown.

In [ ]:
for name in gene_names_sorted:
    gid = name_to_id[name]
    g = sel[sel["gene_id"] == gid].sort_values("query_start")
    q_t = query_bed.get_by_id(f"{gid}_aligned")
    if q_t is None:
        print(f"\n{'='*60}\n{name} ({gid}): no query prediction\n")
        continue

    q_interval = GenomicInterval(q_t.chrom, q_t.start, q_t.end)
    hits = annotation.get_transcripts_in_interval(q_interval)

    print(f"\n{'='*60}")
    print(f"{name} ({gid.replace('U_', '')})")
    print(f"Query: {q_t.chrom}:{q_t.start}-{q_t.end} ({q_t.end - q_t.start:,} bp span)")
    print(f"Islands: {len(g)} ({(g['type']=='anchor').sum()} anchor, {(g['type']=='fill').sum()} fill)")
    print(f"Transcripts in span: {len(hits)}")

    # merge all GENCODE exon blocks and transcript spans (avoid double-counting)
    all_exon_blocks = [t.blocks for t in hits]
    all_spans = np.array([[t.start, t.end] for t in hits]) if len(hits) > 0 else np.empty((0, 2), dtype=int)

    if len(all_exon_blocks) > 0:
        merged_exons = merge_intervals(np.vstack(all_exon_blocks))
        merged_spans = merge_intervals(all_spans)
    else:
        merged_exons = np.empty((0, 2), dtype=int)
        merged_spans = np.empty((0, 2), dtype=int)

    # per-island table with exonic / intronic / intergenic breakdown
    island_rows = []
    for _, isl in g.iterrows():
        isl_arr = np.array([[isl["query_start"], isl["query_end"]]])
        isl_len = isl["query_len"]

        exon_isect = intersect_intervals(isl_arr, merged_exons)
        exonic_bp = int(np.sum(exon_isect[:, 1] - exon_isect[:, 0])) if len(exon_isect) > 0 else 0

        span_isect = intersect_intervals(isl_arr, merged_spans)
        in_transcript_bp = int(np.sum(span_isect[:, 1] - span_isect[:, 0])) if len(span_isect) > 0 else 0

        intronic_bp = in_transcript_bp - exonic_bp
        intergenic_bp = isl_len - in_transcript_bp

        island_rows.append({
            "island": isl["query_island"],
            "type": isl["type"],
            "q_start": isl["query_start"],
            "q_end": isl["query_end"],
            "q_len": isl_len,
            "mmd": f"{isl['diag_mmd']:.4f}",
            "exonic": exonic_bp,
            "intronic": intronic_bp,
            "intergenic": intergenic_bp,
        })
    isl_df = pd.DataFrame(island_rows)
    print(f"\n{isl_df.to_string(index=False)}")

    # plot: genomic layout
    fig, ax = plt.subplots(figsize=(12, 2.5))
    span_start = q_t.start
    span_end = q_t.end

    # predicted blocks (from BED12)
    for block_s, block_e in q_t.blocks:
        ax.barh(1.0, block_e - block_s, left=block_s, height=0.3,
                color="#42A5F5", edgecolor="#1976D2", linewidth=0.5)

    # annotation exons in the region
    anno_y = 0.4
    for t in hits:
        for block_s, block_e in t.blocks:
            if block_e > span_start and block_s < span_end:
                ax.barh(anno_y, block_e - block_s, left=block_s, height=0.2,
                        color="#4CAF50", alpha=0.5, edgecolor="#388E3C", linewidth=0.3)

    # island MMD annotations
    for _, isl in g.iterrows():
        mid = (isl["query_start"] + isl["query_end"]) / 2
        color = "#1565C0" if isl["type"] == "anchor" else "#E65100"
        ax.plot(mid, 1.5, marker="v", color=color, markersize=5)
        ax.text(mid, 1.65, f'{isl["diag_mmd"]:.3f}', ha="center", fontsize=6, color=color)

    ax.set_xlim(span_start - (span_end - span_start) * 0.02,
                span_end + (span_end - span_start) * 0.02)
    ax.set_ylim(0, 2.0)
    ax.set_yticks([0.4, 1.0])
    ax.set_yticklabels(["GENCODE", "Predicted"])
    ax.set_xlabel(f"{q_t.chrom} position")
    ax.set_title(f"{name} — query islands and GENCODE overlap", fontweight="bold")

    legend_elements = [
        mpatches.Patch(color="#42A5F5", label="Predicted blocks"),
        mpatches.Patch(color="#4CAF50", alpha=0.5, label="GENCODE exons"),
        plt.Line2D([0], [0], marker="v", color="#1565C0", linestyle="None", label="Anchor MMD"),
        plt.Line2D([0], [0], marker="v", color="#E65100", linestyle="None", label="Fill MMD"),
    ]
    ax.legend(handles=legend_elements, fontsize=7, loc="upper right")

    plt.tight_layout()
    plt.show()